Imports

In [56]:
import math
import os
import time
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

Utilities for CUDA

In [57]:
def to_device(x):
    return x.cuda() if torch.cuda.is_available() else x

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Masked2D Convolutions

In [58]:
class MaskedConv2d(nn.Conv2d):
    """Correct implementation of masked convolution with both spatial and channel masking"""
    def __init__(self, mask_type, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True):
        super().__init__(in_channels, out_channels, kernel_size, stride, padding, dilation, groups, bias)
        assert mask_type in ('A', 'B'), "mask_type must be 'A' or 'B'"

        # Register mask buffer
        self.register_buffer('mask', torch.ones_like(self.weight.data))

        # Create spatial mask
        kH, kW = kernel_size, kernel_size
        self.mask.fill_(0)
        self.mask[:, :, :kH//2] = 1  # All rows above center
        self.mask[:, :, kH//2, :kW//2] = 1  # Left of center in center row

        if mask_type == 'B':
            self.mask[:, :, kH//2, kW//2] = 1  # Center pixel for mask B

        # Apply channel-wise masking for RGB if input has 3 channels
        if in_channels == 3 and out_channels % 3 == 0:
            out_groups = out_channels // 3
            for i in range(out_groups):
                for j in range(3):
                    # Only allow connections from channels that come before current channel
                    if mask_type == 'A' and j > i:
                        self.mask[i*3+j, :, kH//2, kW//2] = 0
                    elif mask_type == 'B' and j >= i:
                        # For mask B, allow connection to same channel but not future ones
                        if j > i:
                            self.mask[i*3+j, :, kH//2, kW//2] = 0

    def forward(self, x):
        self.weight.data *= self.mask
        return super().forward(x)

PixelCNN Class

In [59]:
class ResidualBlock(nn.Module):
    """Residual block with proper masking"""
    def __init__(self, nr_filters):
        super().__init__()
        self.net = nn.Sequential(
            nn.ReLU(inplace=True),
            MaskedConv2d('B', nr_filters, nr_filters, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(nr_filters, nr_filters, kernel_size=1)
        )

    def forward(self, x):
        return x + self.net(x)

class PixelCNN(nn.Module):
    def __init__(self, in_channels=3, nr_residual=15, nr_filters=128):
        super().__init__()
        # First layer with mask A
        self.first = MaskedConv2d('A', in_channels, nr_filters, kernel_size=7, padding=3)

        # Residual blocks
        self.res_blocks = nn.ModuleList([
            ResidualBlock(nr_filters) for _ in range(nr_residual)
        ])

        # Output layers
        self.final = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(nr_filters, nr_filters, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(nr_filters, in_channels * 256, kernel_size=1)
        )

    def forward(self, x):
        h = self.first(x)
        for block in self.res_blocks:
            h = block(h)
        return self.final(h)



RowLSTM Class

In [60]:

class RowLSTM(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=64, num_layers=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.input_conv = MaskedConv2d('A', in_channels, hidden_dim, kernel_size=7, padding=3)

        # Use LSTMCell for simplicity
        self.lstm_cells = nn.ModuleList([
            nn.LSTMCell(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])

        self.out_conv = nn.Conv2d(hidden_dim, in_channels * 256, kernel_size=1)

    def forward(self, x):
        batch_size, _, height, width = x.shape
        device = x.device

        # Process input
        h_conv = self.input_conv(x)

        # Initialize states
        hidden_states = [torch.zeros(batch_size, self.hidden_dim, width, device=device)
                        for _ in range(self.num_layers)]
        cell_states = [torch.zeros(batch_size, self.hidden_dim, width, device=device)
                      for _ in range(self.num_layers)]

        outputs = []

        for row in range(height):
            current_input = h_conv[:, :, row, :]  # [B, hidden_dim, W]

            # Process through LSTM layers
            for layer in range(self.num_layers):
                # Reshape for LSTMCell: [B*W, hidden_dim]
                input_flat = current_input.permute(0, 2, 1).contiguous().view(-1, self.hidden_dim)
                h_flat = hidden_states[layer].permute(0, 2, 1).contiguous().view(-1, self.hidden_dim)
                c_flat = cell_states[layer].permute(0, 2, 1).contiguous().view(-1, self.hidden_dim)

                # LSTM update
                h_new, c_new = self.lstm_cells[layer](input_flat, (h_flat, c_flat))

                # Reshape back
                h_new = h_new.view(batch_size, width, -1).permute(0, 2, 1)
                c_new = c_new.view(batch_size, width, -1).permute(0, 2, 1)

                hidden_states[layer] = h_new
                cell_states[layer] = c_new
                current_input = h_new

            outputs.append(current_input.unsqueeze(2))

        # Combine outputs
        h_out = torch.cat(outputs, dim=2)
        return self.out_conv(h_out)

Diagonal BiLSTM Class

In [61]:
class DiagonalBiLSTM(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=64, num_layers=3, height=32, width=32):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.height = height
        self.width = width

        # Precompute diagonal indices for faster processing
        self.register_buffer('diag_indices', self._precompute_diagonal_indices(height, width))

        self.input_conv = MaskedConv2d('A', in_channels, hidden_dim, kernel_size=7, padding=3)

        # Use nn.LSTM instead of LSTMCell for better performance
        self.lstm_forward = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.lstm_backward = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.out_conv = nn.Conv2d(hidden_dim, in_channels * 256, kernel_size=1)

    def _precompute_diagonal_indices(self, height, width):
        """Precompute indices for all diagonals"""
        indices = []
        for d in range(height + width - 1):
            diag_indices = []
            for i in range(height):
                j = d - i
                if 0 <= j < width:
                    diag_indices.append((i, j))
            indices.append(diag_indices)
        return indices

    def extract_diagonals(self, x):
        """Extract all diagonals using precomputed indices"""
        batch_size, channels = x.shape[0], x.shape[1]
        diagonals = []

        for diag_indices in self.diag_indices:
            if not diag_indices:
                continue

            # Extract all pixels for this diagonal at once
            rows = [idx[0] for idx in diag_indices]
            cols = [idx[1] for idx in diag_indices]
            diag = x[:, :, rows, cols]  # [B, C, L]
            diagonals.append(diag)

        return diagonals

    def reconstruct_from_diagonals(self, diagonals):
        """Reconstruct 2D feature map from diagonals using precomputed indices"""
        batch_size, channels = diagonals[0].shape[0], diagonals[0].shape[1]
        output = torch.zeros(batch_size, channels, self.height, self.width,
                            device=diagonals[0].device)

        for diag, diag_indices in zip(diagonals, self.diag_indices):
            if not diag_indices:
                continue

            # Place all pixels for this diagonal at once
            rows = [idx[0] for idx in diag_indices]
            cols = [idx[1] for idx in diag_indices]
            output[:, :, rows, cols] = diag

        return output

    def forward(self, x):
        batch_size = x.shape[0]
        device = x.device

        # Process input
        h_conv = self.input_conv(x)

        # Extract diagonals
        diagonals = self.extract_diagonals(h_conv)

        # Process each diagonal
        processed_diagonals = []

        for diag in diagonals:
            # Forward pass - process entire diagonal at once
            diag_forward = diag.permute(0, 2, 1)  # [B, L, C]
            h_forward, _ = self.lstm_forward(diag_forward)
            h_forward = h_forward.permute(0, 2, 1)  # [B, C, L]

            # Backward pass - process reversed diagonal
            diag_backward = torch.flip(diag, [2]).permute(0, 2, 1)  # [B, L, C]
            h_backward, _ = self.lstm_backward(diag_backward)
            h_backward = h_backward.permute(0, 2, 1)  # [B, C, L]
            h_backward = torch.flip(h_backward, [2])  # Reverse back

            # Combine
            processed_diagonals.append(h_forward + h_backward)

        # Reconstruct
        h_out = self.reconstruct_from_diagonals(processed_diagonals)
        return self.out_conv(h_out)

Wrapper Classes

In [62]:
class PixelRNN_PixelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = PixelCNN()

    def forward(self, x):
        return self.model(x)

class PixelRNN_RowLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = RowLSTM()
    def forward(self, x):
        return self.model(x)

class PixelRNN_DiagBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = DiagonalBiLSTM()

    def forward(self, x):
        return self.model(x)

Evaluation Utilities

In [63]:

def logits_to_nll(logits, target):
    """Calculate negative log likelihood with proper reshaping"""
    B, C256, H, W = logits.shape
    C = target.shape[1]
    logits = logits.view(B, C, 256, H, W).permute(0, 1, 3, 4, 2).contiguous()
    logits = logits.view(-1, 256)
    targets = target.view(-1)
    loss = F.cross_entropy(logits, targets, reduction='sum')
    return loss


def nll_to_bits_per_dim(nll, batch_size, C, H, W):
    # bits/dim = (nll / log(2)) / (N * D)
    nll_bits = nll / math.log(2)
    dims = batch_size * C * H * W
    return nll_bits / dims

Dataset Loader

In [64]:

def to_long_tensor(x):
    # torchvision ToTensor gives float in [0,1]
    return (x * 255).long()

def get_dataloaders(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        to_long_tensor
    ])
    train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

    # On Windows, set num_workers=0 to avoid multiprocessing issues
    train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader

def init_weights_xavier(m):
  if isinstance(m, (nn.Conv2d, nn.Linear)):
      nn.init.xavier_uniform_(m.weight)
      if getattr(m, 'bias', None) is not None:
          nn.init.constant_(m.bias, 0)

Train Function

In [65]:


def train_epoch(model, opt, loader, device):
    model.train()
    total_nll = 0.0
    total_samples = 0
    t0 = time.time()
    for batch_idx, (xb, _) in enumerate(loader):
        xb = xb.to(device=device, dtype=torch.long)       # targets
        x_in = (xb.float() / 255.0).to(device)            # normalized inputs [0,1]
        logits = model(x_in)
        loss = logits_to_nll(logits, xb)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        total_nll += loss.item()
        total_samples += xb.size(0)

        # Debug: print first batch loss only
        if batch_idx == 0:
            print("Initial batch loss (nats):", loss.item())

    t1 = time.time()
    return total_nll, total_samples, t1 - t0


Evaluation Function

In [66]:

def eval_epoch(model, loader, device):
    model.eval()
    total_nll = 0.0
    total_samples = 0
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device=device, dtype=torch.long)
            x_in = (xb.float() / 255.0).to(device)        # normalize same as training
            logits = model(x_in)
            loss = logits_to_nll(logits, xb)
            total_nll += loss.item()
            total_samples += xb.size(0)
    return total_nll, total_samples


In [ ]:
# ----------------------------- Notebook Run -------------------------------
# Parameters (can be edited directly in notebook cells)
model_choice = 'diagbilstm'  # 'pixelcnn', 'rowlstm', or 'diagbilstm'
batch_size = 16
epochs = 10
lr = 1e-3

# Dataloaders
train_loader, val_loader = get_dataloaders(batch_size=batch_size)

# Model selection
if model_choice == 'pixelcnn':
    model = PixelRNN_PixelCNN().to(DEVICE)
    model.apply(init_weights_xavier)
elif model_choice == 'rowlstm':
    model = PixelRNN_RowLSTM().to(DEVICE)
else:
    model = PixelRNN_DiagBiLSTM().to(DEVICE)

# Optimizer
# opt = torch.optim.Adam(model.parameters(), lr=lr)
# Use RMSprop as in the paper (common setting)
opt = torch.optim.RMSprop(model.parameters(), lr=lr, alpha=0.95, eps=1e-8)

# # Optional: a scheduler to reduce LR every N epochs
# from torch.optim.lr_scheduler import StepLR
# scheduler = StepLR(opt, step_size=10, gamma=0.5)  # reduce lr by 2 every 10 epochs (tune as needed)

# Training loop
history = {'train_bpd': [], 'val_bpd': []}
for ep in range(epochs):
    t_nll, t_samples, t_time = train_epoch(model, opt, train_loader, DEVICE)

    #scheduler.step()

    train_bpd = nll_to_bits_per_dim(t_nll, t_samples, 3, 32, 32)
    v_nll, v_samples = eval_epoch(model, val_loader, DEVICE)
    val_bpd = nll_to_bits_per_dim(v_nll, v_samples, 3, 32, 32)
    history['train_bpd'].append(train_bpd)
    history['val_bpd'].append(val_bpd)
    print(f"Epoch {ep+1}/{epochs}  train bpd: {train_bpd:.4f}  val bpd: {val_bpd:.4f}  time: {t_time:.1f}s")

# Plot results
plt.plot(history['train_bpd'], label='train bpd')
plt.plot(history['val_bpd'], label='val bpd')
plt.legend()
plt.xlabel('epoch')
plt.ylabel('bits/dim')
plt.title(f"Training {model_choice}")
plt.show()

print('Training complete.')

Initial batch loss (nats): 272763.03125
